In [1]:
# === IMPORTS & CONFIG ===
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from functools import partial

# Allow imports from the src directory
if Path.cwd().name == "notebooks":
    sys.path.insert(0, str(Path.cwd().parent))
else:
    sys.path.insert(0, str(Path.cwd()))

from src.backtest.core import monthly_rebalance
from src.strategies.risk_parity import risk_parity_weights
from src.strategies.momentum import ts_momentum_weights
from src.metrics import cagr, annual_volatility, sharpe, max_drawdown

# --- Configuration ---
DATA_PATH = Path("../data/prices_monthly.csv")
TABLE_DIR = Path("../reports/tables")
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Backtest parameters from config/backtest.yaml
COST_BPS = 10
MAX_WEIGHT = 0.35
LOOKBACK_RP = 12
LOOKBACK_MOM = 12

# === LOAD DATA ===
prices = pd.read_csv(DATA_PATH, parse_dates=["date"]).set_index("date").sort_index()

# === DEFINE STRATEGY FUNCTIONS ===
weight_func_rp = partial(risk_parity_weights, window_months=LOOKBACK_RP)
weight_func_mom = partial(ts_momentum_weights, lookback_months=LOOKBACK_MOM)

# === METRICS HELPER ===
def calculate_metrics(nav):
    returns = nav.pct_change().fillna(0.0)
    return {
        "CAGR": cagr(nav),
        "Volatility": annual_volatility(returns),
        "Sharpe": sharpe(returns),
        "MaxDrawdown": max_drawdown(nav)
    }

# === RUN WALK-FORWARD ANALYSIS ===
results = []
windows = [
    {"start": 2003, "train_end": 2012, "valid_end": 2017, "test_end": 2021},
    {"start": 2005, "train_end": 2014, "valid_end": 2019, "test_end": 2023}
]

print("Running Walk-Forward Analysis...")
for w in windows:
    window_label = f"{w['start']}-{w['test_end']}"
    print(f"  Processing window: {window_label}")
    
    # Define data splits for this window
    valid_prices = prices[(prices.index.year > w['train_end']) & (prices.index.year <= w['valid_end'])]
    test_prices = prices[(prices.index.year > w['valid_end']) & (prices.index.year <= w['test_end'])]

    if len(valid_prices) < 12 or len(test_prices) < 12:
        print(f"    Skipping window {window_label} due to insufficient data.")
        continue

    strategies = {
        "risk_parity": weight_func_rp,
        "ts_momentum": weight_func_mom
    }

    for name, weight_func in strategies.items():
        # Run on validation set
        nav_valid, _, _ = monthly_rebalance(valid_prices, weight_func, max_weight=MAX_WEIGHT, cost_bps=COST_BPS)
        metrics_valid = calculate_metrics(nav_valid)
        results.append({"window": window_label, "split": "valid", "strategy": name, **metrics_valid})

        # Run on test set
        nav_test, _, _ = monthly_rebalance(test_prices, weight_func, max_weight=MAX_WEIGHT, cost_bps=COST_BPS)
        metrics_test = calculate_metrics(nav_test)
        results.append({"window": window_label, "split": "test", "strategy": name, **metrics_test})

# === SAVE RESULTS ===
wf_df = pd.DataFrame(results)
out_path = TABLE_DIR / "walkforward_metrics.csv"
wf_df.to_csv(out_path, index=False)

print("\nSaved walk-forward results to:", out_path)

# Display summary of test results
print("\n--- Walk-Forward Test Set Summary (Mean) ---")
test_results = wf_df[wf_df['split'] == 'test']
summary = test_results.groupby("strategy")[["CAGR", "Sharpe", "MaxDrawdown"]].mean().round(3)
print(summary)

Running Walk-Forward Analysis...
  Processing window: 2003-2021
  Processing window: 2005-2023

Saved walk-forward results to: ../reports/tables/walkforward_metrics.csv

--- Walk-Forward Test Set Summary (Mean) ---
              CAGR  Sharpe  MaxDrawdown
strategy                               
risk_parity  0.018   0.763       -0.027
ts_momentum  0.063   0.759       -0.094
